# Delivery Confirmation

Modelo para identificação de pacotes e assinaturas a partir de imagens em tempo real.

# Prepare environment

## Environment
- DBR 15.4 LTS ML
- 1 x rd-fleet.2xlarge
- Access Mode: Dedicated

## Upgrade python-snappy

In [0]:
%pip install python-snappy --upgrade

## Load parameters

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("database", "")
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("endpoint_name", "")
dbutils.widgets.text("imgs_path", "")

catalog = dbutils.widgets.get("catalog")
database = dbutils.widgets.get("database")
model_name = dbutils.widgets.get("model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")
imgs_path = dbutils.widgets.get("imgs_path")

## Helper functions

In [0]:
def display_detection(img_path, object_name, all_coordinates, all_confidences):
    print(f"Received coordinates: {all_coordinates}")
    try:
        # Load the image
        image = Image.open(img_path)
        
        # Create a figure and axis
        fig, ax = plt.subplots(1)
        
        # Display the image
        ax.imshow(image)

        # Define a color map for different confidence levels
        colors = ['r', 'g', 'b', 'y', 'c', 'm']

        # Create Rectangle patches for each detection
        for coordinates, confidence in zip(all_coordinates, all_confidences):
            color_index = min(int(confidence * len(colors)), len(colors) - 1)
            color = colors[color_index]
            
            # Unpack coordinates
            x1, y1, x2, y2 = coordinates
            
            rect = patches.Rectangle((x1, y1), 
                                     x2 - x1, 
                                     y2 - y1, 
                                     linewidth=2, edgecolor=color, facecolor='none')
            
            # Add the patch to the Axes
            ax.add_patch(rect)

            # Add label
            plt.text(x1, y1 - 10, 
                     f"{object_name} ({confidence:.3f})", 
                     color=color, fontsize=10, weight='bold')

        # Turn off axis
        plt.axis('off')
        
        # Set the title
        plt.title(f"Detected {object_name}s", fontsize=14, fontweight='bold')
        
        # Display the image with the bounding boxes
        plt.show()
    except Exception as e:
        print(f"Error in display_detection: {e}")

# Define the model

In [0]:
import mlflow
import codecs
import io
import torch
from PIL import Image
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd

class OwlViTODModel(mlflow.pyfunc.PythonModel):

  def load_context(self, context):
    self.processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
    self.model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

  def pre_process(self, model_input):
    img_str = model_input.iloc[0, 0]
    image = Image.open(io.BytesIO(codecs.decode(codecs.encode(img_str, "utf-8"), 'base64')))
    return image

  def detect_object(self, image, object_to_find, threshold=0.3):
    # Process the image
    prompt = [[f"a photo of a {object_to_find}"]]
    inputs = self.processor(text=prompt, images=image, return_tensors="pt")
    outputs = self.model(**inputs)

    # Process the output
    target_sizes = torch.Tensor([image.size[::-1]])
    results = self.processor.post_process_object_detection(outputs=outputs, threshold=threshold, target_sizes=target_sizes)

    # Extract results for the first (and only) image
    boxes, scores, labels = results[0]["boxes"], results[0]["scores"], results[0]["labels"]

    # Prepare lists to store all detections
    all_coordinates = []
    all_confidences = []

    # Process all detections
    for box, score in zip(boxes, scores):
      coordinates = [round(i, 2) for i in box.tolist()]
      confidence = round(score.item(), 3)
      all_coordinates.append(coordinates)
      all_confidences.append(confidence)

    # Return all results
    return all_coordinates, all_confidences
  
  def post_process(self, pkg_coordinates, pkg_confidences, sig_coordinates, sig_confidences):
    return pd.DataFrame({
      'pkg_coordinates': [pkg_coordinates],
      'pkg_confidences': [pkg_confidences],
      'sig_coordinates': [sig_coordinates],
      'sig_confidences': [sig_confidences]
    })
    
  def predict(self, context, model_input):
    image = self.pre_process(model_input)
    pkg_coordinates, pkg_confidences = self.detect_object(image, 'package')
    sig_coordinates, sig_confidences = self.detect_object(image, 'signature')
    return self.post_process(pkg_coordinates, pkg_confidences, sig_coordinates, sig_confidences)

# Test the model

In [0]:
display(dbutils.fs.ls(imgs_path))

In [0]:
img_path = f'{imgs_path}/calcular-frete-correios-pac-1024x570.webp'
image = Image.open(img_path)
image

In [0]:
# Convert image to string
byte_array = io.BytesIO()
image.save(byte_array, format='PNG')
img_str = codecs.decode(codecs.encode(byte_array.getvalue(), "base64"), 'utf-8')
img_df = pd.DataFrame({'image': [img_str]})

# Test the model
model = OwlViTODModel()
model.load_context(_)
result = model.predict(_, img_df)

display_detection(img_path, 'package', result['pkg_coordinates'][0], result['pkg_confidences'][0])

# Register the model

In [0]:
spark.sql(f'create database if not exists {catalog}.{database}')

In [0]:
with mlflow.start_run() as run:
  mlflow.pyfunc.log_model(
    "model",
    python_model=OwlViTODModel(),
    input_example=img_df,
    registered_model_name=f'{catalog}.{database}.{model_name}'
  )

# Deploy the model

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput, AutoCaptureConfigInput, ServedModelInputWorkloadSize
from mlflow import MlflowClient

# Find latest model version
mlflow_client = MlflowClient()
latest_model_version = 1
for mv in mlflow_client.search_model_versions(f"name='{catalog}.{database}.{model_name}'"):
    version_int = int(mv.version)
    if version_int > latest_model_version:
        latest_model_version = version_int

# Deploy model to a Model Serving endpoint
w = WorkspaceClient()
endpoint_config = EndpointCoreConfigInput(
    name=endpoint_name,
    served_models=[
        ServedModelInput(
            model_name=f'{catalog}.{database}.{model_name}',
            model_version=latest_model_version,
            workload_size=ServedModelInputWorkloadSize.SMALL,
            scale_to_zero_enabled=True
        )
    ],
    auto_capture_config=AutoCaptureConfigInput(
        catalog_name=catalog,
        schema_name=database,
        enabled=True
    )
)

existing_endpoint = next(
    (e for e in w.serving_endpoints.list() if e.name == endpoint_name), None
)
if existing_endpoint == None:
    print(f"Creating endpoint {endpoint_name} from version {latest_model_version}, this might take a few minutes...")
    w.serving_endpoints.create_and_wait(name=endpoint_name, config=endpoint_config)
else:
    print(f"Updating endpoint {endpoint_name} to version {latest_model_version}, this might take a few minutes...")
    w.serving_endpoints.update_config_and_wait(served_models=endpoint_config.served_models, name=endpoint_name)

# Query endpoint

In [0]:
from PIL import Image

img_path = f'{imgs_path}/calcular-frete-correios-pac-1024x570.webp'
image = Image.open(img_path)
image

In [0]:
from databricks.sdk import WorkspaceClient
import codecs
import io
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import time

# Convert image to string
byte_array = io.BytesIO()
image.save(byte_array, format='PNG')
img_str = codecs.decode(codecs.encode(byte_array.getvalue(), "base64"), 'utf-8')

# Query endpoint
w = WorkspaceClient()
start = time.time()
response = w.serving_endpoints.query(endpoint_name, inputs=[{"image": img_str}])
end = time.time()

result = response.predictions[0]
print(f"Query took {end - start} seconds")
print(result)
display_detection(img_path, 'package', result['pkg_coordinates'], result['pkg_confidences'])

# Evaluate latency

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

n_parallel = 4
n_requests = 100

# Create clients
clients = [WorkspaceClient() for _ in range(n_parallel)]

# Load images
images = []
for f in dbutils.fs.ls(imgs_path):
    image = Image.open(f.path.replace('dbfs:', ''))
    byte_array = io.BytesIO()
    image.save(byte_array, format='PNG')
    img_str = codecs.decode(codecs.encode(byte_array.getvalue(), "base64"), 'utf-8')
    images.append(img_str)
n_images = len(images)

# Define query function
def query_endpoint(n):    
    w = clients[n%n_parallel]
    img_str = images[n%n_images]
    start = time.time()
    response = w.serving_endpoints.query(endpoint_name, inputs=[{"image": img_str}])
    end = time.time()
    return {'predictions': response.predictions[0], 'time': end - start}

# Execute queries in parallel
tasks = list(range(1, n_requests+1))
results = []
start = time.time()
with ThreadPoolExecutor(max_workers=n_parallel) as executor:
    future_to_task = {executor.submit(query_endpoint, t): t for t in tasks}
    for future in as_completed(future_to_task):
        result = future.result()
        results.append(result)
end = time.time()
wall_time = end - start

print(results)

In [0]:
# Extract times from results
times = [r['time'] for r in results]

# Create a pandas DataFrame
df = pd.DataFrame(times, columns=['time'])

# Calculate the mean
avg_time = df['time'].mean()
cpu_time = df['time'].sum()

print(f'''
  RESULTS:
  - Number of requests: {n_requests}
  - Wall time: {wall_time} seconds
  - CPU time: {cpu_time} seconds
  - Avg. concurrency: {cpu_time / wall_time}
  - Avg. latency: {avg_time} seconds
  - Throughput: {n_requests / wall_time} qps
''')

# App

Go to: https://e2-demo-field-eng.cloud.databricks.com/apps/vr-object-detection